<a href="https://colab.research.google.com/github/hfakhimi5058-tech/OC-MSI-Oral-Cancer-AI/blob/main/OC_MSI_Oral_Cancer_AI_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# ORAL CANCER OC-MSI
# ConvNeXt V2 vs DINOv2
# Google Colab - Complete Pipeline
# ============================================================

# -----------------------------
# 1. INSTALL LIBRARIES
# -----------------------------

!pip -q install timm scikit-learn seaborn pandas matplotlib kagglehub

# -----------------------------
# 2. IMPORTS
# -----------------------------

import os
import glob
import copy
import random
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    auc
)

from tqdm.auto import tqdm


# -----------------------------
# 3. RANDOM SEED
# -----------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# -----------------------------
# 4. DEVICE
# -----------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("DEVICE:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled.")
    print("Go to Runtime > Change runtime type > GPU")

print("=" * 60)


# ============================================================
# 5. DOWNLOAD DATASET FROM KAGGLE
# ============================================================

print("\nDownloading OC-MSI dataset...")

import kagglehub

DATASET_PATH = kagglehub.dataset_download(
    "zoya77/oral-cancer-multispectral-imaging-dataset-oc-msi"
)

print("\nDataset downloaded to

SyntaxError: unterminated string literal (detected at line 100) (2357676801.py, line 100)

In [ ]:
# ============================================================
# STEP 18 — INSTALL MODERN VISION MODEL DEPENDENCIES
# SwinV2-Tiny
# ============================================================

!pip -q install timm

In [ ]:
# ============================================================
# STEP 19 — VERIFY GPU + SWINV2-TINY
# ============================================================

import torch
import timm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available.")

# Find available SwinV2 models
swin_models = timm.list_models("*swinv2*")

print("\nAvailable SwinV2 models:")
for name in swin_models[:30]:
    print(" -", name)

print("\nTotal SwinV2 models found:", len(swin_models))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

Available SwinV2 models:
 - swinv2_base_window8_256
 - swinv2_base_window12_192
 - swinv2_base_window12to16_192to256
 - swinv2_base_window12to24_192to384
 - swinv2_base_window16_256
 - swinv2_cr_base_224
 - swinv2_cr_base_384
 - swinv2_cr_base_ns_224
 - swinv2_cr_giant_224
 - swinv2_cr_giant_384
 - swinv2_cr_huge_224
 - swinv2_cr_huge_384
 - swinv2_cr_large_224
 - swinv2_cr_large_384
 - swinv2_cr_small_224
 - swinv2_cr_small_384
 - swinv2_cr_small_ns_224
 - swinv2_cr_small_ns_256
 - swinv2_cr_tiny_224
 - swinv2_cr_tiny_384
 - swinv2_cr_tiny_ns_224
 - swinv2_large_window12_192
 - swinv2_large_window12to16_192to256
 - swinv2_large_window12to24_192to384
 - swinv2_small_window8_256
 - swinv2_small_window16_256
 - swinv2_tiny_window8_256
 - swinv2_tiny_window16_256

Total SwinV2 models found: 28


In [ ]:
# ============================================================
# STEP 20 — LOAD SWINV2-TINY
# ============================================================

import torch
import timm

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# Create SwinV2-Tiny
# ------------------------------------------------------------

swin_model = timm.create_model(
    "swinv2_tiny_window8_256",
    pretrained=True,
    num_classes=2
)

swin_model = swin_model.to(device)


# ------------------------------------------------------------
# Model information
# ------------------------------------------------------------

total_params = sum(p.numel() for p in swin_model.parameters())
trainable_params = sum(
    p.numel() for p in swin_model.parameters()
    if p.requires_grad
)

print("\nModel:", "SwinV2-Tiny")
print("Input size: 256 × 256")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\nSwinV2-Tiny loaded successfully.")

Device: cuda
GPU: Tesla T4


model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            


Model: SwinV2-Tiny
Input size: 256 × 256
Total parameters: 27,579,692
Trainable parameters: 27,579,692

SwinV2-Tiny loaded successfully.


In [ ]:
# ============================================================
# STEP 21 — SWINV2-TINY DATA LOADERS
# ============================================================

import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

DATA_ROOT = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split"

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

print("Train:", TRAIN_DIR)
print("Validation:", VAL_DIR)
print("Test:", TEST_DIR)


# ------------------------------------------------------------
# ImageNet normalization
# SwinV2 pretrained weights expect ImageNet-style normalization
# ------------------------------------------------------------

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# ------------------------------------------------------------
# Training augmentation
# Keep augmentation comparable to EfficientNet experiment
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.92, 1.08)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.05
    ),

    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


# ------------------------------------------------------------
# Validation / Test
# NO augmentation
# ------------------------------------------------------------

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform
)


# ------------------------------------------------------------
# DataLoaders
# ------------------------------------------------------------

BATCH_SIZE_SWIN = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE_SWIN,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE_SWIN,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_SWIN,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ------------------------------------------------------------
# Verify class mapping and sample counts
# ------------------------------------------------------------

print("\nClass mapping:")
print(train_dataset.class_to_idx)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

print("\nNumber of batches:")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

Train: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split/train
Validation: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split/val
Test: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split/test


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split/train'

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder created:")
print(PROJECT_DIR)

Project folder created:
/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI


In [ ]:
import os
import glob

print("Checking current runtime files...\n")

for path in [
    "/kaggle/working",
    "/content",
    "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"
]:
    print(f"\n{path}")
    if os.path.exists(path):
        files = glob.glob(os.path.join(path, "*"))
        print(files[:20])
    else:
        print("Not found")

Checking current runtime files...


/kaggle/working
['/kaggle/working/best_OC_MSI_EfficientNetB0_v2.keras', '/kaggle/working/best_OC_MSI_EfficientNetB0_finetuned.keras', '/kaggle/working/best_OC_MSI_EfficientNetB0.keras', '/kaggle/working/OC_MSI_Split']

/content
['/content/drive', '/content/sample_data']

/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI
[]


In [ ]:
import os
import shutil

PROJECT_DIR = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"

# Create folders
models_dir = os.path.join(PROJECT_DIR, "models")
split_dir = os.path.join(PROJECT_DIR, "OC_MSI_Split")

os.makedirs(models_dir, exist_ok=True)

# Copy trained models
model_files = [
    "/kaggle/working/best_OC_MSI_EfficientNetB0_v2.keras",
    "/kaggle/working/best_OC_MSI_EfficientNetB0_finetuned.keras",
    "/kaggle/working/best_OC_MSI_EfficientNetB0.keras"
]

for src in model_files:
    if os.path.exists(src):
        dst = os.path.join(models_dir, os.path.basename(src))
        shutil.copy2(src, dst)
        print("Saved:", dst)
    else:
        print("NOT FOUND:", src)

# Copy leakage-safe dataset split
src_split = "/kaggle/working/OC_MSI_Split"

if os.path.exists(src_split):
    if os.path.exists(split_dir):
        shutil.rmtree(split_dir)

    shutil.copytree(src_split, split_dir)
    print("\nDataset split saved:", split_dir)
else:
    print("Split folder not found.")

print("\nDONE — project backup completed.")

Saved: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/models/best_OC_MSI_EfficientNetB0_v2.keras
Saved: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/models/best_OC_MSI_EfficientNetB0_finetuned.keras
Saved: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/models/best_OC_MSI_EfficientNetB0.keras

Dataset split saved: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split

DONE — project backup completed.


In [ ]:
import os
import json
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Final locked threshold selected on validation set
FINAL_THRESHOLD = 0.4558

# Final test metrics
results = {
    "model": "EfficientNetB0",
    "task": "Oral Cancer Classification",
    "dataset": "OC-MSI",
    "input_size": "224x224x3",
    "validation_threshold": FINAL_THRESHOLD,
    "test_accuracy": 0.8767,
    "test_sensitivity_cancer": 0.8228,
    "test_specificity": 0.9403,
    "test_precision_cancer": 0.9420,
    "test_f1_cancer": 0.8784,
    "test_macro_f1": 0.8767,
    "validation_roc_auc": 0.9754,
    "validation_pr_auc": 0.9774,
    "validation_sensitivity": 0.9067,
    "validation_specificity": 0.9577,
    "test_samples": 146,
    "test_cancer": 79,
    "test_non_cancer": 67
}

# Save JSON
json_path = os.path.join(RESULTS_DIR, "final_test_results.json")

with open(json_path, "w") as f:
    json.dump(results, f, indent=4)

# Save CSV
csv_path = os.path.join(RESULTS_DIR, "final_test_results.csv")

pd.DataFrame([results]).to_csv(csv_path, index=False)

print("Results saved successfully:")
print(json_path)
print(csv_path)

Results saved successfully:
/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/results/final_test_results.json
/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/results/final_test_results.csv


In [ ]:

import os

PROJECT_DIR = "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI"

readme = """# OC-MSI Oral Cancer AI

## AI-Assisted Oral Cancer Classification Using EfficientNetB0

This repository contains a research-oriented deep learning pipeline for
binary classification of oral cancer and non-cancer images using the
OC-MSI dataset and EfficientNetB0.

### Researcher

**Haleh Fakhimi Rezaei**

Dentistry | Genetics | Biotechnology | Artificial Intelligence in Oral Cancer Research

### Model

- Architecture: EfficientNetB0
- Transfer learning: ImageNet pretrained weights
- Input size: 224 × 224 × 3
- Data augmentation: rotation, translation, zoom, horizontal flip, brightness variation
- Fine-tuning: top layers of EfficientNetB0
- Binary classification: Cancer vs Non-Cancer

### Dataset

OC-MSI (Oral Cancer Multispectral Imaging Dataset)

Dataset composition:

- Total images: 950
- Cancer: 500
- Non-Cancer: 450

Exact duplicate images were identified and grouped to reduce the risk of
data leakage between train, validation, and test sets.

### Data Split

- Training: 658 images
- Validation: 146 images
- Test: 146 images

### Validation Performance

- ROC-AUC: 0.9754
- PR-AUC: 0.9774
- Sensitivity: 90.67%
- Specificity: 95.77%

The classification threshold was selected using the validation set only.

### Held-Out Test Performance

- Accuracy: 87.67%
- Cancer sensitivity: 82.28%
- Specificity: 94.03%
- Cancer precision: 94.20%
- Cancer F1-score: 87.84%

The test set was evaluated using a fixed threshold selected from the
validation set.

### Important Scientific Notes

This project is a research prototype for image classification and is
not a clinical diagnostic system.

Exact duplicate grouping was performed, but patient-level independence
could not be confirmed from the available dataset metadata. Therefore,
patient-level generalization should not be assumed.

The dataset is referred to as multispectral by its source; the image
files used in this project were processed as RGB images.

### Reproducibility

The repository is intended to document the complete experimental workflow,
including dataset inspection, leakage-aware splitting, model training,
fine-tuning, validation-based threshold selection, and held-out test
evaluation.

### Author

Haleh Fakhimi Rezaei
"""

readme_path = os.path.join(PROJECT_DIR, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)

print("README created:")
print(readme_path)

README created:
/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/README.md


In [ ]:
# ============================================================
# STEP 25 — SWINV2-TINY FINE-TUNING
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import timm

from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 1. Load BEST Stage-1 model
# ------------------------------------------------------------

swin_model = timm.create_model(
    "swinv2_tiny_window8_256",
    pretrained=False,
    num_classes=2
)

swin_model.load_state_dict(
    torch.load(
        "/content/best_OC_MSI_SwinV2_Tiny_stage1.pth",
        map_location=device
    )
)

swin_model = swin_model.to(device)


# ------------------------------------------------------------
# 2. Freeze everything first
# ------------------------------------------------------------

for param in swin_model.parameters():
    param.requires_grad = False


# ------------------------------------------------------------
# 3. Unfreeze last part of backbone
# ------------------------------------------------------------

children = list(swin_model.children())

print("Model children:")

for i, child in enumerate(children):
    print(i, type(child).__name__)


# ------------------------------------------------------------
# 4. Unfreeze final layers
# ------------------------------------------------------------

for param in swin_model.layers[-1].parameters():
    param.requires_grad = True

for param in swin_model.norm.parameters():
    param.requires_grad = True

for param in swin_model.head.parameters():
    param.requires_grad = True


# ------------------------------------------------------------
# 5. Loss
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# 6. Optimizer
# ------------------------------------------------------------

optimizer = optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        swin_model.parameters()
    ),
    lr=1e-5,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# 7. Count trainable parameters
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in swin_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in swin_model.parameters()
    if p.requires_grad
)

print("\nFine-tuning setup:")
print("Total parameters:", f"{total_params:,}")
print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)
print(
    "Frozen parameters:",
    f"{total_params - trainable_params:,}"
)

print("Learning rate:", 1e-5)

NameError: name 'device' is not defined

In [ ]:
from pathlib import Path
import json
import re
from datetime import datetime

root = Path("/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI")

print("=== فایل‌های مدل، تنظیمات و گزارش‌ها ===")
wanted = {".keras", ".h5", ".csv", ".json", ".txt", ".log", ".pkl"}

if root.exists():
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix.lower() in wanted:
            s = p.stat()
            modified = datetime.fromtimestamp(s.st_mtime).isoformat(
                sep=" ", timespec="seconds"
            )
            print(
                f"{p.relative_to(root)} | "
                f"{s.st_size / 1024**2:.2f} MB | modified: {modified}"
            )
else:
    print("پوشه پروژه در این مسیر پیدا نشد.")

notebooks = [
    Path("/content/drive/MyDrive/Colab Notebooks/OC_MSI_Oral_Cancer_AI (1).ipynb"),
    Path("/content/drive/MyDrive/Colab Notebooks/oral cancer.ipynb"),
]

pattern = re.compile(
    r"EfficientNet|preprocess_input|rescale|image_dataset|"
    r"ImageDataGenerator|flow_from|train_test_split|"
    r"GroupShuffleSplit|StratifiedGroupKFold|"
    r"\.fit\s*\(|\.compile\s*\(|load_model|"
    r"ModelCheckpoint|EarlyStopping|CSVLogger|"
    r"trainable|initial_epoch|class_names|class_indices",
    re.I,
)

for nb_path in notebooks:
    if not nb_path.exists():
        continue

    print(f"\n=== نوت‌بوک: {nb_path.name} ===")
    nb = json.loads(nb_path.read_text(encoding="utf-8"))

    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        if not pattern.search(source):
            continue

        print(f"\n--- Cell {i} ---")
        print(source[:7000])
        if len(source) > 7000:
            print("[کد این سلول کوتاه شده است]")

        # نمایش انتهای خروجی ذخیره‌شده، برای یافتن اپوک‌ها و نتایج
        outputs = []
        for out in cell.get("outputs", []):
            text = out.get("text", [])
            if not text:
                text = out.get("data", {}).get("text/plain", [])
            outputs.append(text if isinstance(text, str) else "".join(text))

        saved_output = "\n".join(outputs)
        if saved_output.strip():
            print("\nخروجی ذخیره‌شده — بخش پایانی:")
            print(saved_output[-3500:])


=== فایل‌های مدل، تنظیمات و گزارش‌ها ===
پوشه پروژه در این مسیر پیدا نشد.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive")

extensions = {".ipynb", ".keras", ".h5", ".hdf5", ".pt", ".pth"}
keywords = ("oral", "oc-msi", "oc_msi", "efficientnet", "checkpoint")

for p in root.rglob("*"):
    if p.is_file():
        if (
            p.suffix.lower() in extensions
            or p.name == "checkpoint"
            or any(k in p.name.lower() for k in keywords)
        ):
            print(p)


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/Copy of Untitled0 (3).ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Untitled0 (2).ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Untitled0 (1).ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/ToothFairy2_3D_Dental_AI.ipynb
/content/drive/MyDrive/Colab Notebooks/01_load_and_inspect_toothfairy2.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Dental research Studio.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Dental research Studio (2).ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Dental research Studio (1).ipynb
/content/drive/MyDrive/Colab Notebooks/Another copy of Dental research Studio.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Dental research Studio.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/d

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI")

print("=== PROJECT ===")
print("Found:", PROJECT.is_dir())

print("\n=== MODELS ===")
models_dir = PROJECT / "models"
models = sorted(models_dir.glob("*.keras"))

if not models:
    print("No .keras models found.")

for p in models:
    print(f"{p.name} | {p.stat().st_size / (1024**2):.1f} MB")

print("\n=== DATASET ===")
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

for split in ["train", "val", "test"]:
    folder = PROJECT / "OC_MSI_Split" / split

    if not folder.is_dir():
        print(f"{split}: NOT FOUND")
        continue

    for cls in sorted(p for p in folder.iterdir() if p.is_dir()):
        count = sum(
            1 for p in cls.rglob("*")
            if p.is_file() and p.suffix.lower() in extensions
        )
        print(f"{split} / {cls.name}: {count}")

print("\n=== SAVED RESULTS ===")
result_file = PROJECT / "results" / "final_test_results.json"

if result_file.is_file():
    print(result_file.read_text(encoding="utf-8"))
else:
    print("Results file not found.")


=== PROJECT ===
Found: True

=== MODELS ===
best_OC_MSI_EfficientNetB0.keras | 18.2 MB
best_OC_MSI_EfficientNetB0_finetuned.keras | 18.2 MB
best_OC_MSI_EfficientNetB0_v2.keras | 18.2 MB

=== DATASET ===
train / MSI_Cancer: 346
train / MSI_Non Cancer: 312
val / MSI_Cancer: 75
val / MSI_Non Cancer: 71
test / MSI_Cancer: 79
test / MSI_Non Cancer: 67

=== SAVED RESULTS ===
{
    "model": "EfficientNetB0",
    "task": "Oral Cancer Classification",
    "dataset": "OC-MSI",
    "input_size": "224x224x3",
    "validation_threshold": 0.4558,
    "test_accuracy": 0.8767,
    "test_sensitivity_cancer": 0.8228,
    "test_specificity": 0.9403,
    "test_precision_cancer": 0.942,
    "test_f1_cancer": 0.8784,
    "test_macro_f1": 0.8767,
    "validation_roc_auc": 0.9754,
    "validation_pr_auc": 0.9774,
    "validation_sensitivity": 0.9067,
    "validation_specificity": 0.9577,
    "test_samples": 146,
    "test_cancer": 79,
    "test_non_cancer": 67
}


In [ ]:
# فقط نصب وابستگی‌ها؛ هنوز مدل دانلود یا آموزش داده نمی‌شود
%pip -q install -U "transformers>=4.56.0,<5" huggingface_hub safetensors scikit-learn pillow

import torch
import transformers
from transformers import AutoImageProcessor, AutoModel

print("=== VERSIONS ===")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("\n=== GPU ===")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total GPU memory: {memory_gb:.1f} GB")
else:
    print("GPU not available.")
    print("Colab: Runtime → Change runtime type → T4 GPU")

print("\n=== DINOv3 SUPPORT ===")
try:
    from transformers import DINOv3ViTModel
    print("DINOv3 ViT support: OK")
except ImportError as e:
    print("DINOv3 import failed:", e)
    print("ممکن است نیاز به Restart session باشد؛ فعلاً سلول بعدی را اجرا نکن.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
=== VERSIONS ===
PyTorch: 2.14.1+cu130
Transformers: 4.57.6

=== GPU ===
GPU: Tesla T4
Total GPU memory: 14.6 GB

=== DINOv3 SUPPORT ===
DINOv3 ViT support: OK


In [ ]:
%pip install -U \
    "transformers>=5,<6" \
    "huggingface-hub>=1.32.0,<2" \
    "pillow>=10,<13" \
    safetensors scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 120.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 52.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 114.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 98.7 MB/s  0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-13.0.0:
      Successfully uninstalled pillow-13.0.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [ ]:
import torch
import transformers
import huggingface_hub
import PIL

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Pillow:", PIL.__version__)

print("\nGPU:",
      torch.cuda.get_device_name(0)
      if torch.cuda.is_available()
      else "Not available")

try:
    from transformers import (
        AutoImageProcessor,
        AutoModel,
        DINOv3ViTModel,
    )
    print("\nDINOv3 support: OK")
except ImportError as e:
    print("\nImport error:", e)


PyTorch: 2.14.1+cu130
Transformers: 5.23.0
Hugging Face Hub: 1.33.0
Pillow: 12.3.0

GPU: Tesla T4

DINOv3 support: OK


In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel

MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = AutoImageProcessor.from_pretrained(MODEL_ID)

dinov3 = AutoModel.from_pretrained(MODEL_ID)
dinov3 = dinov3.to(device)

# فعلاً فقط استخراج ویژگی؛ وزن‌های backbone تغییر نمی‌کنند
dinov3.requires_grad_(False)
dinov3.eval()

print("=== DINOv3 LOADED ===")
print("Model:", MODEL_ID)
print("Device:", device)
print("Parameters:", f"{sum(p.numel() for p in dinov3.parameters()):,}")
print("Feature dimension:", dinov3.config.hidden_size)
print("Trainable parameters:",
      sum(p.numel() for p in dinov3.parameters() if p.requires_grad))

print("\n=== OFFICIAL PREPROCESSING ===")
print(processor)


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m.
401 Client Error. (Request ID: Root=1-6af81b7e-1854bf74437a7f0409102b74;472b4970-5fe5-453a-bec4-4ba0a59811f6)

Cannot access gated repo for url https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/114c1379950215c8b35dfcd4e90a5c251dde0d32/processor_config.json.
Access to model facebook/dinov3-vits16-pretrain-lvd1689m is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI")

print("Project found:", PROJECT_DIR.is_dir())
print("Models folder:", (PROJECT_DIR / "models").is_dir())
print("Dataset folder:", (PROJECT_DIR / "OC_MSI_Split").is_dir())

print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "Not available"
)


Project found: True
Models folder: True
Dataset folder: True
GPU: Tesla T4


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("GPU:",
      torch.cuda.get_device_name(0)
      if torch.cuda.is_available()
      else "Not available")

try:
    import transformers
    import huggingface_hub
    from transformers import AutoImageProcessor, AutoModel, DINOv3ViTModel

    print("Transformers:", transformers.__version__)
    print("Hugging Face Hub:", huggingface_hub.__version__)
    print("DINOv3 support: OK")
except ImportError as e:
    print("Import error:", e)


PyTorch: 2.14.1+cu130
GPU: Tesla T4
Transformers: 5.22.1
Hugging Face Hub: 1.33.0
DINOv3 support: OK


In [4]:
from huggingface_hub import notebook_login

notebook_login()


In [5]:
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError

MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"

try:
    config_path = hf_hub_download(
        repo_id=MODEL_ID,
        filename="config.json",
        token=True,
    )
    print("Access OK — دسترسی مدل تأیید شد.")

except GatedRepoError:
    print(
        "دسترسی مدل هنوز برقرار نیست. "
        "وضعیت درخواست در صفحهٔ مدل و مجوز خواندن توکن را بررسی کن."
    )

except HfHubHTTPError as e:
    print("HTTP status:", e.response.status_code)


دسترسی مدل هنوز برقرار نیست. وضعیت درخواست در صفحهٔ مدل و مجوز خواندن توکن را بررسی کن.


In [1]:
from google.colab import drive
from pathlib import Path
import tensorflow as tf

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI")
MODEL_PATH = (
    PROJECT_DIR / "models" /
    "best_OC_MSI_EfficientNetB0_finetuned.keras"
)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Model exists:", MODEL_PATH.is_file())

assert MODEL_PATH.is_file(), f"مدل پیدا نشد: {MODEL_PATH}"

effnet_model = tf.keras.models.load_model(
    str(MODEL_PATH),
    compile=False
)

print("\n=== MODEL ===")
print("Input shape:", effnet_model.input_shape)
print("Output shape:", effnet_model.output_shape)
print("Parameters:", f"{effnet_model.count_params():,}")

effnet_model.summary()

print("\n=== PREPROCESSING / AUGMENTATION LAYERS ===")

def inspect_preprocessing(model, prefix=""):
    for layer in model.layers:
        layer_type = type(layer).__name__
        layer_path = f"{prefix}{layer.name}"

        if any(word in layer_type.lower() for word in [
            "rescaling", "normalization", "resizing",
            "cropping", "random", "lambda"
        ]):
            print(f"\n{layer_path} | {layer_type}")
            print(layer.get_config())

        if isinstance(layer, tf.keras.Model):
            inspect_preprocessing(layer, prefix=layer_path + "/")

inspect_preprocessing(effnet_model)

print("\n=== LAST LAYERS ===")
for layer in effnet_model.layers[-6:]:
    activation = getattr(layer, "activation", None)
    activation_name = getattr(activation, "__name__", None)

    print(
        f"{layer.name} | {type(layer).__name__} | "
        f"trainable={layer.trainable} | activation={activation_name}"
    )


Mounted at /content/drive
TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Model exists: True

=== MODEL ===
Input shape: (None, 224, 224, 3)
Output shape: (None, 1)
Parameters: 4,218,788


Model: "OC_MSI_EfficientNetB0_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_128 (Dense)               │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ non_cancer_probability (Dense)  │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,218,788 (16.09 MB)

 Trainable params: 166,657 (651.00 KB)

 Non-trainable params: 4,052,131 (15.46 MB)


=== PREPROCESSING / AUGMENTATION LAYERS ===

efficientnetb0/rescaling_2 | Rescaling
{'name': 'rescaling_2', 'trainable': False, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'scale': 0.00392156862745098, 'offset': 0.0}

efficientnetb0/normalization_1 | Normalization
{'name': 'normalization_1', 'trainable': False, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'axis': (3,), 'invert': False, 'mean': None, 'variance': None}

efficientnetb0/rescaling_3 | Rescaling
{'name': 'rescaling_3', 'trainable': False, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'scale': [2.0896918976428642, 2.1128856368212916, 2.1081851067789197], 'offset': 0.0}

efficientnetb0/stem_bn | BatchNormalization
{'name': 'stem_bn', 'trainable': False, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config':

In [2]:
import tensorflow as tf

print("=== INPUT / OUTPUT ===")
print("Input:", effnet_model.input_shape)
print("Output:", effnet_model.output_shape)

print("\n=== TOP-LEVEL LAYERS ===")
for layer in effnet_model.layers:
    print(
        layer.name,
        "|", type(layer).__name__,
        "| trainable:", layer.trainable
    )

print("\n=== PREPROCESSING ONLY (excluding BatchNormalization) ===")

def inspect_compact(model, prefix=""):
    for layer in model.layers:
        kind = type(layer).__name__
        path = prefix + layer.name

        if kind in {
            "Rescaling", "Normalization", "Resizing",
            "CenterCrop", "Lambda"
        } or kind.startswith("Random"):
            config = layer.get_config()
            # حذف اطلاعات عمومی غیرضروری
            for key in ["name", "trainable", "dtype"]:
                config.pop(key, None)
            print(path, "|", kind, "|", config)

        if isinstance(layer, tf.keras.Model):
            inspect_compact(layer, path + "/")

inspect_compact(effnet_model)

print("\n=== BACKBONE STATUS ===")
for layer in effnet_model.layers:
    if isinstance(layer, tf.keras.Model):
        convs = [
            sub for sub in layer.layers
            if isinstance(sub, (
                tf.keras.layers.Conv2D,
                tf.keras.layers.DepthwiseConv2D
            ))
        ]
        print("Nested model:", layer.name)
        print("Convolution layers:", len(convs))
        print("Trainable convolution layers:",
              sum(sub.trainable for sub in convs))
        print("Trainable weight tensors:", len(layer.trainable_weights))

print("\n=== OUTPUT LAYER ===")
last = effnet_model.layers[-1]
print("Name:", last.name)
print("Units:", getattr(last, "units", None))
print("Activation:",
      getattr(getattr(last, "activation", None), "__name__", None))


=== INPUT / OUTPUT ===
Input: (None, 224, 224, 3)
Output: (None, 1)

=== TOP-LEVEL LAYERS ===
image_input | InputLayer | trainable: True
efficientnetb0 | Functional | trainable: False
global_average_pooling | GlobalAveragePooling2D | trainable: True
batch_normalization | BatchNormalization | trainable: True
dropout | Dropout | trainable: True
dense_128 | Dense | trainable: True
dropout_2 | Dropout | trainable: True
non_cancer_probability | Dense | trainable: True

=== PREPROCESSING ONLY (excluding BatchNormalization) ===
efficientnetb0/rescaling_2 | Rescaling | {'scale': 0.00392156862745098, 'offset': 0.0}
efficientnetb0/normalization_1 | Normalization | {'axis': (3,), 'invert': False, 'mean': None, 'variance': None}
efficientnetb0/rescaling_3 | Rescaling | {'scale': [2.0896918976428642, 2.1128856368212916, 2.1081851067789197], 'offset': 0.0}

=== BACKBONE STATUS ===
Nested model: efficientnetb0
Convolution layers: 81
Trainable convolution layers: 0
Trainable weight tensors: 0

=== OUT

In [3]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI")

# پوشه‌های داده و وزن‌ها را برای جست‌وجوی کد پیمایش نمی‌کنیم
import os

skip_dirs = {
    "OC_MSI_Split", "models", ".git",
    "__pycache__", ".ipynb_checkpoints"
}

wanted_extensions = {".ipynb", ".py", ".json", ".csv", ".yaml", ".yml"}
candidate_files = []

for root, dirs, files in os.walk(PROJECT_DIR):
    dirs[:] = sorted(d for d in dirs if d not in skip_dirs)

    for name in sorted(files):
        path = Path(root) / name
        if path.suffix.lower() in wanted_extensions:
            candidate_files.append(path)

print("=== PROJECT CODE / METADATA FILES ===")
for path in candidate_files[:100]:
    print(path.relative_to(PROJECT_DIR))

print("\nTotal files found:", len(candidate_files))

print("\n=== NOTEBOOKS IN DEFAULT COLAB FOLDER ===")
colab_dir = Path("/content/drive/MyDrive/Colab Notebooks")

if colab_dir.is_dir():
    notebooks = sorted(colab_dir.glob("*.ipynb"))
    for path in notebooks[:40]:
        print(path.name)
    print("Total notebooks:", len(notebooks))
else:
    print("Default Colab Notebooks folder not found.")


=== PROJECT CODE / METADATA FILES ===
results/final_test_results.csv
results/final_test_results.json

Total files found: 2

=== NOTEBOOKS IN DEFAULT COLAB FOLDER ===
01_load_and_inspect_toothfairy2.ipynb
Another copy of Dental research Studio.ipynb
Copy of Dental research Studio (1).ipynb
Copy of Dental research Studio (2).ipynb
Copy of Dental research Studio.ipynb
Copy of Untitled0 (1).ipynb
Copy of Untitled0 (2).ipynb
Copy of Untitled0 (3).ipynb
Copy of Untitled0.ipynb
Dental research Studio.ipynb
OC_MSI_Oral_Cancer_AI (1).ipynb
ToothFairy2_3D_Dental_AI.ipynb
Untitled0.ipynb
Untitled1.ipynb
Untitled2.ipynb
oral cancer.ipynb
Total notebooks: 16


In [4]:
import json
import re
from pathlib import Path

NOTEBOOK_DIR = Path("/content/drive/MyDrive/Colab Notebooks")

notebook_names = [
    "OC_MSI_Oral_Cancer_AI (1).ipynb",
    "oral cancer.ipynb",
]

pattern = re.compile(
    r"class_indices|class_names|class_mode|"
    r"flow_from_directory|image_dataset_from_directory|"
    r"ImageDataGenerator|preprocess_input|"
    r"rescale\s*=|/\s*255|classes\s*=|"
    r"label_mode|MSI_Cancer|MSI_Non Cancer|"
    r"convert_to_tensor|decode_image",
    re.IGNORECASE,
)

for name in notebook_names:
    path = NOTEBOOK_DIR / name
    print(f"\n{'=' * 65}\nNOTEBOOK: {name}\n{'=' * 65}")

    if not path.is_file():
        print("Not found")
        continue

    with path.open("r", encoding="utf-8") as f:
        notebook = json.load(f)

    printed_lines = 0
    matched_cells = 0

    for cell_number, cell in enumerate(notebook.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", [])
        code = source if isinstance(source, str) else "".join(source)

        # حذف سلول‌های ابزار بررسی خودمان از نتایج جست‌وجو
        if any(marker in code for marker in [
            "notebook_names =", "inspect_preprocessing",
            "inspect_compact", "candidate_files"
        ]):
            continue

        lines = code.splitlines()
        hits = [i for i, line in enumerate(lines) if pattern.search(line)]

        if not hits:
            continue

        matched_cells += 1
        print(f"\n--- Code cell {cell_number} ---")

        # نمایش سه خط قبل و بعد هر تطبیق
        selected = set()
        for i in hits:
            selected.update(range(max(0, i - 3), min(len(lines), i + 4)))

        for i in sorted(selected):
            print(f"{i + 1:03d}: {lines[i]}")
            printed_lines += 1
            if printed_lines >= 180:
                break

        if printed_lines >= 180:
            print("\n[Output limit reached for this notebook]")
            break

    if matched_cells == 0:
        print("No matching code found.")



NOTEBOOK: OC_MSI_Oral_Cancer_AI (1).ipynb

--- Code cell 5 ---
024: swin_model = timm.create_model(
025:     "swinv2_tiny_window8_256",
026:     pretrained=True,
027:     num_classes=2
028: )
029: 
030: swin_model = swin_model.to(device)

--- Code cell 13 ---
019: swin_model = timm.create_model(
020:     "swinv2_tiny_window8_256",
021:     pretrained=False,
022:     num_classes=2
023: )
024: 
025: swin_model.load_state_dict(

--- Code cell 14 ---
028: ]
029: 
030: pattern = re.compile(
031:     r"EfficientNet|preprocess_input|rescale|image_dataset|"
032:     r"ImageDataGenerator|flow_from|train_test_split|"
033:     r"GroupShuffleSplit|StratifiedGroupKFold|"
034:     r"\.fit\s*\(|\.compile\s*\(|load_model|"
035:     r"ModelCheckpoint|EarlyStopping|CSVLogger|"
036:     r"trainable|initial_epoch|class_names|class_indices",
037:     re.I,
038: )
039: 

NOTEBOOK: oral cancer.ipynb

--- Code cell 4 ---
028: ]
029: 
030: pattern = re.compile(
031:     r"EfficientNet|preprocess_input|rescale

In [5]:
import json
import re
from pathlib import Path

folder = Path("/content/drive/MyDrive/Colab Notebooks")

signals = {
    "EfficientNet": r"EfficientNetB0|best_OC_MSI|non_cancer_probability",
    "OC-MSI data": r"OC_MSI_Split|MSI_Cancer|MSI_Non Cancer",
    "Data loader": r"flow_from_directory|image_dataset_from_directory",
    "Training": r"\.fit\s*\(",
}

# سلول‌های جست‌وجو/بازرسی قبلی را کنار می‌گذاریم
inspection_markers = [
    "re.compile(", "json.load(", "inspect_compact",
    "inspect_preprocessing", "candidate_files",
    "notebook_names =", "signals =",
]

found = False

for path in sorted(folder.glob("*.ipynb")):
    try:
        with path.open(encoding="utf-8") as f:
            nb = json.load(f)
    except Exception as e:
        print(f"Cannot read {path.name}: {type(e).__name__}")
        continue

    matches = []
    for number, cell in enumerate(nb.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", [])
        code = source if isinstance(source, str) else "".join(source)

        if any(marker in code for marker in inspection_markers):
            continue

        tags = [
            label for label, expression in signals.items()
            if re.search(expression, code, flags=re.I)
        ]
        if tags:
            matches.append((number, tags))

    if matches:
        found = True
        print(f"\nNOTEBOOK: {path.name}")
        for number, tags in matches:
            print(f"  Cell {number}: {', '.join(tags)}")

if not found:
    print("No relevant code found in this folder.")



NOTEBOOK: Copy of Untitled0 (1).ipynb
  Cell 5: OC-MSI data
  Cell 9: EfficientNet, OC-MSI data
  Cell 10: EfficientNet
  Cell 11: EfficientNet
  Cell 12: EfficientNet

NOTEBOOK: Copy of Untitled0 (3).ipynb
  Cell 7: OC-MSI data
  Cell 8: OC-MSI data
  Cell 10: OC-MSI data
  Cell 11: OC-MSI data
  Cell 12: OC-MSI data
  Cell 13: OC-MSI data
  Cell 14: OC-MSI data
  Cell 15: OC-MSI data, Data loader
  Cell 16: EfficientNet
  Cell 17: EfficientNet, Training
  Cell 18: EfficientNet, OC-MSI data, Data loader
  Cell 19: EfficientNet
  Cell 20: EfficientNet
  Cell 21: EfficientNet, Training
  Cell 22: EfficientNet, Training

NOTEBOOK: Copy of Untitled0.ipynb
  Cell 2: OC-MSI data
  Cell 6: EfficientNet
  Cell 7: OC-MSI data
  Cell 8: OC-MSI data
  Cell 14: OC-MSI data
  Cell 15: OC-MSI data
  Cell 17: OC-MSI data
  Cell 18: OC-MSI data
  Cell 19: OC-MSI data
  Cell 20: OC-MSI data
  Cell 21: OC-MSI data
  Cell 22: OC-MSI data, Data loader
  Cell 23: EfficientNet
  Cell 24: EfficientNet, Tra

In [6]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

# شماره‌ها مطابق فهرست جست‌وجوی قبلی، شامل تمام انواع سلول هستند.
for number, cell in enumerate(notebook.get("cells", []), start=1):
    if 15 <= number <= 22 and cell.get("cell_type") == "code":
        source = cell.get("source", [])
        code = source if isinstance(source, str) else "".join(source)

        print("\n" + "=" * 70)
        print(f"CELL {number}")
        print("=" * 70)
        print(code)



CELL 15

# ============================================================
# STEP 8 — DATA PIPELINE
# Transfer Learning Preparation
# ============================================================

import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

base_dir = "/kaggle/working/OC_MSI_Split"

train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")
test_dir  = os.path.join(base_dir, "test")


# ------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 16
SEED = 42


# ------------------------------------------------------------
# TRAIN AUGMENTATION
# ------------------------------------------------------------

train_datagen = ImageDataGenerator(

    resc

In [7]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

cell = nb["cells"][14]  # سلول شماره ۱۵
source = cell.get("source", [])
code = source if isinstance(source, str) else "".join(source)

print("=== CELL 15: SOURCE ONLY ===")
print(code)


=== CELL 15: SOURCE ONLY ===

# ============================================================
# STEP 8 — DATA PIPELINE
# Transfer Learning Preparation
# ============================================================

import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

base_dir = "/kaggle/working/OC_MSI_Split"

train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")
test_dir  = os.path.join(base_dir, "test")


# ------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 16
SEED = 42


# ------------------------------------------------------------
# TRAIN AUGMENTATION
# ------------------------------------------------------------

train_datagen = ImageData

In [8]:
  {'MSI_Cancer': 0, 'MSI_Non Cancer': 1}


{'MSI_Cancer': 0, 'MSI_Non Cancer': 1}

In [9]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

source = nb["cells"][14].get("source", [])
code = source if isinstance(source, str) else "".join(source)
lines = code.splitlines()

print(f"Cell 15 — total lines: {len(lines)}")
print("=== FIRST 60 LINES ===")

for number, line in enumerate(lines[:60], start=1):
    print(f"{number:03d}: {line}")


Cell 15 — total lines: 152
=== FIRST 60 LINES ===
001: 
002: # ============================================================
003: # STEP 8 — DATA PIPELINE
004: # Transfer Learning Preparation
005: # ============================================================
006: 
007: import os
008: import tensorflow as tf
009: from tensorflow.keras.preprocessing.image import ImageDataGenerator
010: 
011: # ------------------------------------------------------------
012: # PATHS
013: # ------------------------------------------------------------
014: 
015: base_dir = "/kaggle/working/OC_MSI_Split"
016: 
017: train_dir = os.path.join(base_dir, "train")
018: val_dir   = os.path.join(base_dir, "val")
019: test_dir  = os.path.join(base_dir, "test")
020: 
021: 
022: # ------------------------------------------------------------
023: # PARAMETERS
024: # ------------------------------------------------------------
025: 
026: IMG_HEIGHT = 224
027: IMG_WIDTH = 224
028: BATCH_SIZE = 16
029: SEED = 42
030: 
031

In [10]:
rescale=1.0 / 255.0


In [11]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

source = nb["cells"][17].get("source", [])
code = source if isinstance(source, str) else "".join(source)
lines = code.splitlines()

print(f"Cell 18 — total lines: {len(lines)}")
print("=== FIRST 100 LINES — SOURCE ONLY ===")

for number, line in enumerate(lines[:100], start=1):
    print(f"{number:03d}: {line}")


Cell 18 — total lines: 142
=== FIRST 100 LINES — SOURCE ONLY ===
001: 
002: # ============================================================
003: # STEP 11 — CORRECT EFFICIENTNET INPUT PIPELINE
004: # ============================================================
005: 
006: import os
007: import tensorflow as tf
008: from tensorflow.keras.preprocessing.image import ImageDataGenerator
009: 
010: print("\n" + "=" * 70)
011: print("STEP 11 — REBUILDING EFFICIENTNET DATA PIPELINE")
012: print("=" * 70)
013: 
014: # ------------------------------------------------------------
015: # PARAMETERS
016: # ------------------------------------------------------------
017: 
018: IMG_HEIGHT = 224
019: IMG_WIDTH = 224
020: BATCH_SIZE = 16
021: SEED = 42
022: 
023: base_dir = "/kaggle/working/OC_MSI_Split"
024: 
025: train_dir = os.path.join(base_dir, "train")
026: val_dir   = os.path.join(base_dir, "val")
027: test_dir  = os.path.join(base_dir, "test")
028: 
029: 
030: # ---------------------------------

In [12]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

source = nb["cells"][21].get("source", [])
code = source if isinstance(source, str) else "".join(source)
lines = code.splitlines()

print(f"Cell 22 — total lines: {len(lines)}")
print("=== FIRST 100 LINES — SOURCE ONLY ===")

for number, line in enumerate(lines[:100], start=1):
    print(f"{number:03d}: {line}")


Cell 22 — total lines: 149
=== FIRST 100 LINES — SOURCE ONLY ===
001: # ============================================================
002: # STEP 14 — FINE-TUNING TOP LAYERS OF EFFICIENTNETB0
003: # ============================================================
004: 
005: import tensorflow as tf
006: from tensorflow import keras
007: 
008: print("\n" + "=" * 70)
009: print("STEP 14 — FINE-TUNING EFFICIENTNETB0")
010: print("=" * 70)
011: 
012: # ------------------------------------------------------------
013: # 1. Freeze all layers first
014: # ------------------------------------------------------------
015: 
016: base_model.trainable = True
017: 
018: # ------------------------------------------------------------
019: # 2. Freeze most of the backbone
020: #    Only the top ~30 layers will be trainable
021: # ------------------------------------------------------------
022: 
023: fine_tune_at = len(base_model.layers) - 30
024: 
025: for layer in base_model.layers[:fine_tune_at]:
026:   

In [13]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

source = nb["cells"][20].get("source", [])
code = source if isinstance(source, str) else "".join(source)
lines = code.splitlines()

print(f"Cell 21 — total lines: {len(lines)}")
print("=== FIRST 140 LINES — SOURCE ONLY ===")

for number, line in enumerate(lines[:140], start=1):
    print(f"{number:03d}: {line}")


Cell 21 — total lines: 93
=== FIRST 140 LINES — SOURCE ONLY ===
001: # ============================================================
002: # STEP 13 — INITIAL TRAINING
003: # ============================================================
004: 
005: import os
006: import tensorflow as tf
007: from tensorflow import keras
008: 
009: print("\n" + "=" * 70)
010: print("STEP 13 — TRAINING FRESH EFFICIENTNETB0")
011: print("=" * 70)
012: 
013: # ------------------------------------------------------------
014: # 1. Paths
015: # ------------------------------------------------------------
016: 
017: checkpoint_path = "/kaggle/working/best_OC_MSI_EfficientNetB0_v2.keras"
018: 
019: # ------------------------------------------------------------
020: # 2. Callbacks
021: # ------------------------------------------------------------
022: 
023: checkpoint = keras.callbacks.ModelCheckpoint(
024:     filepath=checkpoint_path,
025:     monitor="val_auc",
026:     mode="max",
027:     save_best_only=True,

In [14]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

source = nb["cells"][19].get("source", [])
code = source if isinstance(source, str) else "".join(source)
lines = code.splitlines()

print(f"Cell 20 — total lines: {len(lines)}")
print("=== FIRST 140 LINES — SOURCE ONLY ===")

for number, line in enumerate(lines[:140], start=1):
    print(f"{number:03d}: {line}")


Cell 20 — total lines: 111
=== FIRST 140 LINES — SOURCE ONLY ===
001: # ============================================================
002: # STEP 12 — BUILD FRESH EFFICIENTNETB0
003: # ============================================================
004: 
005: import tensorflow as tf
006: from tensorflow import keras
007: from tensorflow.keras import layers
008: from tensorflow.keras.applications import EfficientNetB0
009: 
010: print("\n" + "=" * 70)
011: print("STEP 12 — BUILDING FRESH EFFICIENTNETB0")
012: print("=" * 70)
013: 
014: # ------------------------------------------------------------
015: # 1. Build EfficientNetB0
016: # ------------------------------------------------------------
017: 
018: base_model = EfficientNetB0(
019:     weights="imagenet",
020:     include_top=False,
021:     input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
022: )
023: 
024: # Freeze backbone
025: base_model.trainable = False
026: 
027: # ------------------------------------------------------------
028: # 2. Cl

In [15]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

source = nb["cells"][19].get("source", [])
code = source if isinstance(source, str) else "".join(source)

print("=== CELL 20 — LINES 1–53 — SOURCE ONLY ===")
for number, line in enumerate(code.splitlines()[:53], start=1):
    print(f"{number:03d}: {line}")


=== CELL 20 — LINES 1–53 — SOURCE ONLY ===
001: # ============================================================
002: # STEP 12 — BUILD FRESH EFFICIENTNETB0
003: # ============================================================
004: 
005: import tensorflow as tf
006: from tensorflow import keras
007: from tensorflow.keras import layers
008: from tensorflow.keras.applications import EfficientNetB0
009: 
010: print("\n" + "=" * 70)
011: print("STEP 12 — BUILDING FRESH EFFICIENTNETB0")
012: print("=" * 70)
013: 
014: # ------------------------------------------------------------
015: # 1. Build EfficientNetB0
016: # ------------------------------------------------------------
017: 
018: base_model = EfficientNetB0(
019:     weights="imagenet",
020:     include_top=False,
021:     input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
022: )
023: 
024: # Freeze backbone
025: base_model.trainable = False
026: 
027: # ------------------------------------------------------------
028: # 2. Classification head
029:

In [16]:
from pathlib import Path
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

val_dir = Path(
    "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/"
    "OC_MSI_Split/val"
)
assert val_dir.is_dir(), f"Validation folder not found: {val_dir}"

# مدل ذخیره‌شده‌ای که قبلاً بررسی کردیم؛ بارگذاری یا آموزش مجدد ندارد
assert effnet_model.input_shape == (None, 224, 224, 3)
assert effnet_model.output_shape == (None, 1)

val_check = ImageDataGenerator().flow_from_directory(
    str(val_dir),
    classes=["MSI_Cancer", "MSI_Non Cancer"],
    target_size=(224, 224),
    color_mode="rgb",
    batch_size=16,
    class_mode="binary",
    shuffle=False,
    interpolation="nearest",  # پیش‌فرض مسیر آموزش قبلی
)

assert val_check.samples == 146, (
    f"Unexpected validation count: {val_check.samples}"
)
print("Class mapping:", val_check.class_indices)

val_check.reset()
p_non_cancer = effnet_model.predict(val_check, verbose=1).reshape(-1)

assert len(p_non_cancer) == val_check.samples
assert np.isfinite(p_non_cancer).all()
assert ((p_non_cancer >= 0) & (p_non_cancer <= 1)).all()

# تبدیل برچسب‌ها برای گزارش سرطان به‌عنوان کلاس مثبت
y_cancer = (val_check.classes == 0).astype(int)
p_cancer = 1.0 - p_non_cancer
pred_cancer = (p_cancer >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_cancer, pred_cancer, labels=[0, 1]
).ravel()

print("\n=== VALIDATION ONLY | CANCER POSITIVE | THRESHOLD 0.5 ===")
print(f"Accuracy:           {accuracy_score(y_cancer, pred_cancer):.4f}")
print(f"Cancer sensitivity: {recall_score(y_cancer, pred_cancer, zero_division=0):.4f}")
print(f"Specificity:        {tn / (tn + fp):.4f}")
print(f"Cancer precision:   {precision_score(y_cancer, pred_cancer, zero_division=0):.4f}")
print(f"Cancer F1:          {f1_score(y_cancer, pred_cancer, zero_division=0):.4f}")
print(f"Cancer ROC-AUC:     {roc_auc_score(y_cancer, p_cancer):.4f}")
print(f"Cancer AP:          {average_precision_score(y_cancer, p_cancer):.4f}")
print("AP = Average Precision; not trapezoidal PR-AUC.")
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print("Probability range:", float(p_cancer.min()), float(p_cancer.max()))


Found 146 images belonging to 2 classes.
Class mapping: {'MSI_Cancer': 0, 'MSI_Non Cancer': 1}
10/10 ━━━━━━━━━━━━━━━━━━━━ 88s 6s/step

=== VALIDATION ONLY | CANCER POSITIVE | THRESHOLD 0.5 ===
Accuracy:           0.9110
Cancer sensitivity: 0.8533
Specificity:        0.9718
Cancer precision:   0.9697
Cancer F1:          0.9078
Cancer ROC-AUC:     0.9754
Cancer AP:          0.9776
AP = Average Precision; not trapezoidal PR-AUC.
TN=69, FP=2, FN=11, TP=64
Probability range: 5.650520324707031e-05 0.9998005628585815


In [17]:
# ============================================================
# CORRECTED EFFICIENTNETB0 FINE-TUNING — NO TEST ACCESS
# ============================================================
import json
from pathlib import Path
from datetime import datetime
import uuid
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
keras.utils.set_random_seed(SEED)

PROJECT = Path("/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI")
SOURCE = PROJECT / "models/best_OC_MSI_EfficientNetB0_v2.keras"
TRAIN_DIR = PROJECT / "OC_MSI_Split/train"
VAL_DIR = PROJECT / "OC_MSI_Split/val"

for path in [SOURCE, TRAIN_DIR, VAL_DIR]:
    assert path.exists(), f"Not found: {path}"

RUN_DIR = PROJECT / "results" / (
    "corrected_finetune_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
    + "_" + uuid.uuid4().hex[:6]
)
RUN_DIR.mkdir(parents=True, exist_ok=False)
BEST_PATH = RUN_DIR / "best_model.keras"

# 1) Data — matches the corrected training pipeline
train_data = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.08,
    horizontal_flip=True,
    brightness_range=(0.95, 1.05),
    fill_mode="reflect",
).flow_from_directory(
    str(TRAIN_DIR),
    classes=["MSI_Cancer", "MSI_Non Cancer"],
    target_size=(224, 224),
    batch_size=16,
    class_mode="binary",
    shuffle=True,
    seed=SEED,
    interpolation="nearest",
)

val_data = ImageDataGenerator().flow_from_directory(
    str(VAL_DIR),
    classes=["MSI_Cancer", "MSI_Non Cancer"],
    target_size=(224, 224),
    batch_size=16,
    class_mode="binary",
    shuffle=False,
    interpolation="nearest",
)

assert train_data.samples == 658, "Unexpected training count"
assert val_data.samples == 146, "Unexpected validation count"
assert train_data.class_indices == val_data.class_indices

# 2) Load an independent model; preserve previously loaded models
ft_model = keras.models.load_model(str(SOURCE), compile=False)

# Critical fix: obtain backbone from THIS loaded model
backbone = ft_model.get_layer("efficientnetb0")
backbone.trainable = True

cutoff = max(0, len(backbone.layers) - 30)

for i, layer in enumerate(backbone.layers):
    layer.trainable = (
        i >= cutoff
        and not isinstance(layer, keras.layers.BatchNormalization)
    )

# Verify actual trainable convolution weights before training
trainable_convs = [
    layer.name for layer in backbone.layers
    if isinstance(
        layer,
        (keras.layers.Conv2D, keras.layers.DepthwiseConv2D)
    )
    and layer.trainable
    and len(layer.trainable_weights) > 0
]

assert backbone.trainable
assert len(backbone.trainable_weights) > 0
assert trainable_convs, "STOP: no convolution weights are trainable"

print("\nTrainable backbone convolutions:", trainable_convs)
print("Trainable backbone parameters:",
      sum(int(np.prod(w.shape)) for w in backbone.trainable_weights))
print("Output directory:", RUN_DIR)

# 3) Metrics: invert both label and probability to make cancer positive.
# Standard built-in metrics are retained for easy model loading.
@keras.utils.register_keras_serializable(package="OCMSI")
class CancerAUC(keras.metrics.AUC):
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, y_pred.dtype)
        return super().update_state(
            1.0 - y_true, 1.0 - y_pred, sample_weight=sample_weight
        )

@keras.utils.register_keras_serializable(package="OCMSI")
class CancerRecall(keras.metrics.Recall):
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, y_pred.dtype)
        return super().update_state(
            1.0 - y_true, 1.0 - y_pred, sample_weight=sample_weight
        )

ft_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        CancerAUC(name="cancer_auc"),
        CancerRecall(name="cancer_sensitivity"),
    ],
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(BEST_PATH),
        monitor="val_cancer_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_cancer_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_cancer_auc",
        mode="max",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(RUN_DIR / "training_log.csv")),
    keras.callbacks.TerminateOnNaN(),
]

config = {
    "source_model": str(SOURCE),
    "class_indices": train_data.class_indices,
    "external_rescaling": False,
    "last_backbone_layers_considered": 30,
    "backbone_batchnorm_frozen": True,
    "trainable_convolutions": trainable_convs,
    "learning_rate": 1e-5,
    "batch_size": 16,
    "max_epochs": 15,
    "selection_metric": "val_cancer_auc",
    "seed": SEED,
    "tensorflow_version": tf.__version__,
    "patient_separation_verified": False,
}
with (RUN_DIR / "run_config.json").open("w") as f:
    json.dump(config, f, indent=2)

# 4) Train — validation only, no test evaluation
history_corrected = ft_model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks,
    verbose=1,
)

with (RUN_DIR / "history.json").open("w") as f:
    json.dump(
        {key: [float(v) for v in values]
         for key, values in history_corrected.history.items()},
        f,
        indent=2,
    )

best_epoch = int(np.argmax(
    history_corrected.history["val_cancer_auc"]
))

print("\n=== BEST VALIDATION-AUC EPOCH ===")
print("Epoch:", best_epoch + 1)
for key in ["val_loss", "val_accuracy",
            "val_cancer_auc", "val_cancer_sensitivity"]:
    print(f"{key}: {history_corrected.history[key][best_epoch]:.4f}")
print("Saved best model:", BEST_PATH)


Found 658 images belonging to 2 classes.
Found 146 images belonging to 2 classes.

Trainable backbone convolutions: ['block6d_dwconv', 'block6d_se_reduce', 'block6d_se_expand', 'block6d_project_conv', 'block7a_expand_conv', 'block7a_dwconv', 'block7a_se_reduce', 'block7a_se_expand', 'block7a_project_conv', 'top_conv']
Trainable backbone parameters: 1483360
Output directory: /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/results/corrected_finetune_20261115_062440_8652fa
Epoch 1/15
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8981 - cancer_auc: 0.9694 - cancer_sensitivity: 0.8884 - loss: 0.2269
Epoch 1: val_cancer_auc improved from None to 0.97465, saving model to /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/results/corrected_finetune_20261115_062440_8652fa/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/results/corrected_finetune_20261115_062440_8652fa/best_model.keras
42/42 ━━━━━━━━━━━━━━━━━━━━ 154s 3s/step - accuracy: 0.9043 - canc

In [18]:
from pathlib import Path
import numpy as np
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score,
    average_precision_score,
)

checkpoint = Path(
    "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/results/"
    "corrected_finetune_20261115_062440_8652fa/best_model.keras"
)
val_dir = Path(
    "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split/val"
)

# compile=False: no need to reconstruct custom training metrics
checked_model = keras.models.load_model(str(checkpoint), compile=False)

validation = ImageDataGenerator().flow_from_directory(
    str(val_dir),
    classes=["MSI_Cancer", "MSI_Non Cancer"],
    target_size=(224, 224),
    batch_size=16,
    class_mode="binary",
    shuffle=False,
    interpolation="nearest",
)
assert validation.samples == 146

validation.reset()
p_non_cancer = checked_model.predict(validation, verbose=1).reshape(-1)
assert len(p_non_cancer) == validation.samples
assert np.isfinite(p_non_cancer).all()
assert ((p_non_cancer >= 0) & (p_non_cancer <= 1)).all()

y = (validation.classes == 0).astype(int)
p = 1.0 - p_non_cancer
pred = (p >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

print("\n=== CORRECTED MODEL | VALIDATION | THRESHOLD 0.5 ===")
print(f"Accuracy:           {accuracy_score(y, pred):.4f}")
print(f"Cancer sensitivity: {recall_score(y, pred, zero_division=0):.4f}")
print(f"Specificity:        {tn / (tn + fp):.4f}")
print(f"Cancer precision:   {precision_score(y, pred, zero_division=0):.4f}")
print(f"Cancer F1:          {f1_score(y, pred, zero_division=0):.4f}")
print(f"Cancer ROC-AUC:     {roc_auc_score(y, p):.4f}")
print(f"Cancer AP:          {average_precision_score(y, p):.4f}")
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")


Found 146 images belonging to 2 classes.
10/10 ━━━━━━━━━━━━━━━━━━━━ 15s 775ms/step

=== CORRECTED MODEL | VALIDATION | THRESHOLD 0.5 ===
Accuracy:           0.9315
Cancer sensitivity: 0.8933
Specificity:        0.9718
Cancer precision:   0.9710
Cancer F1:          0.9306
Cancer ROC-AUC:     0.9748
Cancer AP:          0.9772
TN=69, FP=2, FN=8, TP=67


In [19]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

for number, cell in enumerate(nb.get("cells", []), start=1):
    if not (7 <= number <= 14) or cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    code = source if isinstance(source, str) else "".join(source)

    print("\n" + "=" * 60)
    print(f"CELL {number}")
    print("=" * 60)
    print(code)



CELL 7

# ============================================================
# STEP 2 — CHECK OC-MSI DATASET
# ============================================================

import os
import glob
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 60)
print("DATASET PATH")
print("=" * 60)
print(DATASET_PATH)


# ------------------------------------------------------------
# 1. SHOW FOLDER STRUCTURE
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FOLDER STRUCTURE")
print("=" * 60)

for root, dirs, files in os.walk(DATASET_PATH):

    level = root.replace(DATASET_PATH, "").count(os.sep)
    indent = "  " * level

    print(indent + os.path.basename(root) + "/")

    for file in files[:10]:
        print(indent + "   " + file)


# ------------------------------------------------------------
# 2. FIND ALL IMAGES
# ------------------------------------------------------------

extensions = [
    "*.jpg",
    "*.jpeg",

In [20]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

source = nb["cells"][12].get("source", [])
code = source if isinstance(source, str) else "".join(source)
lines = code.splitlines()

print(f"Cell 13 — total lines: {len(lines)}")
print("=== FIRST 100 LINES — SOURCE ONLY ===")

for number, line in enumerate(lines[:100], start=1):
    print(f"{number:03d}: {line}")


Cell 13 — total lines: 215
=== FIRST 100 LINES — SOURCE ONLY ===
001: 
002: # ============================================================
003: # STEP 7 — LEAKAGE-SAFE TRAIN / VALIDATION / TEST SPLIT
004: # ============================================================
005: 
006: import os
007: import hashlib
008: import shutil
009: from collections import defaultdict
010: from sklearn.model_selection import train_test_split
011: 
012: dataset_path = "/kaggle/input/oral-cancer-multispectral-imaging-dataset-oc-msi/Multispectral_MSI_Dataset"
013: 
014: output_path = "/kaggle/working/OC_MSI_Split"
015: 
016: classes = ["MSI_Non Cancer", "MSI_Cancer"]
017: 
018: image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
019: 
020: 
021: # ============================================================
022: # STEP 1 — COLLECT IMAGES BY CONTENT HASH
023: # ============================================================
024: 
025: hash_groups = defaultdict(list)
026: 
027: for class_name i

In [21]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Copy of Untitled0 (3).ipynb"
)

with path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

source = nb["cells"][12].get("source", [])
code = source if isinstance(source, str) else "".join(source)
lines = code.splitlines()

print("=== CELL 13 — LINES 98–175 — SOURCE ONLY ===")
for number, line in enumerate(lines[97:175], start=98):
    print(f"{number:03d}: {line}")


=== CELL 13 — LINES 98–175 — SOURCE ONLY ===
098: temp_labels = [
099:     group_labels[i]
100:     for i in temp_idx
101: ]
102: 
103: val_idx, test_idx = train_test_split(
104:     temp_idx,
105:     test_size=0.50,
106:     stratify=temp_labels,
107:     random_state=42
108: )
109: 
110: 
111: print("\n" + "=" * 70)
112: print("GROUP SPLIT")
113: print("=" * 70)
114: 
115: print("Train groups:", len(train_idx))
116: print("Validation groups:", len(val_idx))
117: print("Test groups:", len(test_idx))
118: 
119: 
120: # ============================================================
121: # STEP 4 — CREATE OUTPUT DIRECTORIES
122: # ============================================================
123: 
124: if os.path.exists(output_path):
125:     shutil.rmtree(output_path)
126: 
127: for split in ["train", "val", "test"]:
128:     for class_name in classes:
129: 
130:         os.makedirs(
131:             os.path.join(
132:                 output_path,
133:                 split,
134:         

In [22]:
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/OC_MSI_Oral_Cancer_AI/OC_MSI_Split"
)
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

for split in ["train", "val"]:
    for class_name in ["MSI_Cancer", "MSI_Non Cancer"]:
        folder = root / split / class_name
        assert folder.is_dir(), f"Missing folder: {folder}"

        files = sorted(
            p.name for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in extensions
        )

        print(f"\n=== {split} / {class_name} | {len(files)} files ===")
        for name in files[:12]:
            print(name)



=== train / MSI_Cancer | 346 files ===
001.jpeg
002.jpeg
003.jpeg
004.jpeg
005.jpeg
006.jpeg
007.jpeg
008.jpeg
009.jpeg
010.jpeg
011.jpeg
013.jpeg

=== train / MSI_Non Cancer | 312 files ===
001.jpeg
002.jpeg
003.jpeg
004.jpeg
005.jpeg
006.jpeg
007.jpeg
008.jpeg
010.jpeg
012.jpeg
013.jpeg
014.jpeg

=== val / MSI_Cancer | 75 files ===
012.jpeg
014.jpeg
017.jpeg
023.jpeg
029.jpeg
035.jpeg
041.jpeg
042.jpeg
065.jpeg
069.jpeg
070.jpeg
077.jpeg

=== val / MSI_Non Cancer | 71 files ===
009.jpeg
011.jpeg
015.jpeg
021.jpeg
048.jpeg
060.jpeg
065.jpeg
071.jpeg
078.jpeg
084.jpeg
096.jpeg
101.jpeg


In [25]:
columns = [
    "relative_filename",
    "true_label",
    "p_cancer",
    "predicted_label_at_0_5",
    "model_name",
    "split",
]
